In [23]:
import os
import time
import dask.array as da
from dask.distributed import Client
import numpy as np

In [24]:
def readImg(path):
    img = imageio.imread(path)
    return np.array(img, dtype='uint8')

def writeImg(path, buf):
    imageio.imwrite(path, buf)

def part_median_filter(block):
    # Dimensions de la partie de l'image (bloc)
    nx, ny, _ = block.shape

    # Nouvelle image filtrée
    new_block = np.zeros_like(block)

    # Appliquer le filtre médian sur chaque canal (R, G, B)
    def get_neighbors(x, y, channel):
        neighbors = []
        for i in range(-1, 2):
            for j in range(-1, 2):
                xi = min(max(x + i, 0), nx - 1)
                yi = min(max(y + j, 0), ny - 1)
                neighbors.append(block[xi, yi, channel])
        return np.median(neighbors)

    # Appliquer le filtre médian pour chaque pixel et chaque canal
    for i in range(nx):
        for j in range(ny):
            for c in range(3):  # 0 -> Red, 1 -> Green, 2 -> Blue
                new_block[i, j, c] = get_neighbors(i, j, c)

    return new_block

In [26]:
def main():
    data_dir = 'data'
    file = os.path.join(data_dir, 'lena_noisy.jpg')

    # Load the image (replace with actual image reading function)
    img_buf = readImg(file)  
    print('SHAPE', img_buf.shape)
    nx, ny, _ = img_buf.shape  # Extract dimensions
    nb_partitions = 8
    print("NB PARTITIONS: ", nb_partitions)

    # Split the image into parts (Dask will handle the parallelization)
    block_size = nx // nb_partitions
    partitions = []
    begin = 0
    for ip in range(nb_partitions):
        end = min(begin + block_size, nx)
        partitions.append([ip, begin, end, img_buf])
        begin = end

    # Initialize Dask client
    client = Client()

    # Dask parallel computation
    start_time = time.time()
    results = client.map(part_median_filter, [block[3][block[1]:block[2], :, :] for block in partitions])  # Apply filter in parallel
    result_data = client.gather(results)  # Collect results
    end_time = time.time()

    print(f'Execution Time is : {end_time - start_time} seconds')

    # Reconstruct the new image from the result data
    new_img_buf = np.zeros((nx, ny, 3), dtype='uint8')  # Image with 3 channels (R, G, B)
    for part_id, part_buf in enumerate(result_data):
        first = partitions[part_id][1]
        end = partitions[part_id][2]
        new_img_buf[first:end, :, :] = part_buf

    # Write the filtered image
    print('CREATE NEW PICTURE FILE')
    filter_file = os.path.join(data_dir, 'lena_filter_dask.jpg')
    writeImg(filter_file, new_img_buf)  # Replace with your actual image writing function
    print('IMAGE CREATED SUCCESSFULLY !')

    client.close()  # Close the Dask client

if __name__ == '__main__':
    main()

SHAPE (128, 128, 3)
NB PARTITIONS:  8


/tmp/ipykernel_1834/1096187660.py:2: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(path)
/usr/local/lib/python3.9/dist-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44475 instead
  warnings.warn(


Execution Time is : 0.6222293376922607 seconds
CREATE NEW PICTURE FILE
IMAGE CREATED SUCCESSFULLY !
